# 作业 01：一个预测函数，接收多个输入张量

上周的预测函数只有一个输入 x。这次同一个样本有两个输入 x1、x2，它们共同决定一个目标 y：

$$\hat y = w_1x_1 + w_2x_2 + b.$$

重点：调用一次预测函数时，需要同时传入 x1 和 x2。它们不是两次独立预测，也不是两个不同样本；例如同一个物体的两个测量值，都可能影响预测结果。

下面还会把多个样本分别放进 x1、x2 和 y。`x1[i]`、`x2[i]`、`y[i]` 属于同一个样本。请区分“一个样本有两个输入”和“共有多少个样本”。

本题仅需 CPU 和 PyTorch。训练步骤沿用上周：预测、损失、清空梯度、反向传播、更新参数。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b

# 同一个样本的两个输入，预测时必须一起使用
x1_demo = torch.tensor(1.0)
x2_demo = torch.tensor(2.0)
y_demo = torch.tensor(-3.0)
w1_demo = torch.tensor(1.0)
w2_demo = torch.tensor(1.0)
b_demo = torch.tensor(0.0)
y_pred_demo = predict(x1_demo, x2_demo, w1_demo, w2_demo, b_demo)
loss_demo = torch.abs(y_demo - y_pred_demo)
for name, value in [('x1', x1_demo), ('x2', x2_demo), ('y', y_demo),
                    ('y_pred', y_pred_demo), ('loss', loss_demo)]:
    print(name, 'shape =', tuple(value.shape), 'value =', value)



x1 shape = () value = tensor(1.)
x2 shape = () value = tensor(2.)
y shape = () value = tensor(-3.)
y_pred shape = () value = tensor(3.)
loss shape = () value = tensor(6.)


## 同一个样本，也可以只用一个 x 张量

上面的 x1_demo 和 x2_demo 分别是两个标量张量，shape 都为 `()`。现在直接用 `torch.tensor([1.0, 2.0])` 创建一个张量 x，shape 就是 `(2,)`：**一个样本，包含两个输入值**。这里的 2 表示两个特征，不是两个样本。

预测公式和参数都不变，只改变函数接收输入的方式：原来传入 x1、x2；现在传入 x，再用 x[0]、x[1] 取出两个值。此处只比较单个样本，不进行训练；多样本的训练代码放在作业 02。


In [2]:
# 直接创建一个包含两个输入值的张量，表示同一个样本
x = torch.tensor([1.0, 2.0])

def predict_packed(x, w1, w2, b):
    return w1 * x[0] + w2 * x[1] + b

y_pred_packed = predict_packed(x, w1_demo, w2_demo, b_demo)
loss_packed = torch.abs(y_demo - y_pred_packed)

for name, value in [('x1_demo', x1_demo), ('x2_demo', x2_demo),
                    ('x', x), ('x[0]', x[0]), ('x[1]', x[1]),
                    ('y', y_demo), ('y_pred_packed', y_pred_packed),
                    ('loss_packed', loss_packed)]:
    print(name, 'shape =', tuple(value.shape), 'value =', value)

print('两个输入张量的预测：', y_pred_demo)
print('一个打包张量的预测：', y_pred_packed)
assert x.shape == (2,)
assert torch.allclose(y_pred_demo, y_pred_packed)
assert torch.allclose(loss_demo, loss_packed)


x1_demo shape = () value = tensor(1.)
x2_demo shape = () value = tensor(2.)
x shape = (2,) value = tensor([1., 2.])
x[0] shape = () value = tensor(1.)
x[1] shape = () value = tensor(2.)
y shape = () value = tensor(-3.)
y_pred_packed shape = () value = tensor(3.)
loss_packed shape = () value = tensor(6.)
两个输入张量的预测： tensor(3.)
一个打包张量的预测： tensor(3.)


## 改变一个输入值，比较两种写法

先预测：只把第二个输入加 1，两种写法的预测是否仍然相同？第一个输入是否改变？

下面始终使用同一组参数，不进行训练。用 clone 复制 x 后再修改，以保留原始样本，方便比较。


In [3]:
x_changed = x.clone()
x_changed[1] = x_changed[1] + 1.0

y_pred_separate_changed = predict(
    x1_demo, x2_demo + 1.0, w1_demo, w2_demo, b_demo)
y_pred_packed_changed = predict_packed(
    x_changed, w1_demo, w2_demo, b_demo)

print('原始 x：', x, 'shape：', tuple(x.shape))
print('修改后 x：', x_changed, 'shape：', tuple(x_changed.shape))
print('修改前的预测：', y_pred_packed)
print('修改后，两个输入张量的预测：', y_pred_separate_changed)
print('修改后，一个打包张量的预测：', y_pred_packed_changed)
assert torch.allclose(y_pred_separate_changed, y_pred_packed_changed)


原始 x： tensor([1., 2.]) shape： (2,)
修改后 x： tensor([1., 3.]) shape： (2,)
修改前的预测： tensor(3.)
修改后，两个输入张量的预测： tensor(4.)
修改后，一个打包张量的预测： tensor(4.)


## 我的观察

1. 两个标量张量的 shape 分别是什么？打包后的 x 的 shape 是什么？x[0] 和 x[1] 的 shape 又是什么？（20分）

`x1_demo` 和 `x2_demo` 都是标量张量，shape 都是 `()`，也就是 0 维，维度里连长度都没有。打包后的 `x = torch.tensor([1.0, 2.0])` 的 shape 是 `(2,)`，是 1 维、长度为 2 的张量。`x[0]` 和 `x[1]` 是从 `x` 里取出来的单个元素，取出来之后又变回标量张量，shape 仍然是 `()`。

所以这两种写法正好是一对互逆操作：打包是把两个 `()` 装进一个 `(2,)`，用 `x[i]` 索引是把 `(2,)` 再拆回 `()`。上面单元格的输出可以直接对上：`x1 shape = ()`、`x2 shape = ()`、`x shape = (2,)`、`x[0] shape = ()`、`x[1] shape = ()`。

2. x 的 shape 为 `(2,)`，在本例中代表两个样本，还是一个样本的两个输入值？（10分）

代表**一个样本的两个输入值**，不是两个样本。

这里的 2 指的是特征个数：`x[0]=1.0` 喂给 $w_1$，`x[1]=2.0` 喂给 $w_2$，两个值一起代入同一个公式 $\hat y=w_1x_1+w_2x_2+b$，最后只算出**一个**预测值 `y_pred_packed`，也只对应**一个**目标 `y_demo = -3.0`。

判断的办法是看这个维度最终"算出几个预测"：本例算出 1 个，所以 2 是输入值的个数。如果它真是两个样本，`predict` 就会返回 shape 为 `(2,)` 的两个预测值，也就需要两个 y 来算损失。对比作业 02：那里 `x1`、`x2`、`y` 的 shape 都是 `(6,)`，`predict` 返回 6 个预测值，那个 6 才是样本数。同样是 1 维张量，装的是"一个样本的多个特征"还是"多个样本"，要看它进入公式后产出几个结果。

3. 比较 predict 与 predict_packed：数据的组织和传入方式如何改变？预测公式、参数、y 和 loss 是否改变？（20分）

**改变的两处：**

- 数据的组织：原来一个样本的两个输入分别放在 `x1_demo`、`x2_demo` 两个**独立**的标量张量里，一共两个容器；现在放在**同一个** `x` 张量的两个位置上，容器从两个 `()` 变成一个 `(2,)`。
- 传入方式：`predict(x1, x2, w1, w2, b)` 要传 5 个实参，函数体里直接用形参 `x1`、`x2`；`predict_packed(x, w1, w2, b)` 只传 4 个实参，函数体里必须先用 `x[0]`、`x[1]` 把两个值取出来才能用。改输入值时也一样：原来直接写 `x2_demo + 1.0` 就行，打包版要先 `x.clone()` 复制一份再改 `x_changed[1]`，用 clone 是为了保留原始 `x` 方便前后比较（直接改 `x[1]` 会把原样本也改掉）。

**没变的四处：**

- 预测公式没变：都是"$w_1$ 乘第一个输入 + $w_2$ 乘第二个输入 + $b$"，变的只是输入从形参直接拿还是从 `x` 里索引出来。
- 参数没变：两次都用同一组 `w1_demo=1.0`、`w2_demo=1.0`、`b_demo=0.0`，参数个数仍然是 3 个。参数怎么建、有几个，跟输入怎么打包没有关系。
- y 没变：都是 `y_demo = -3.0`，因为样本本身没变，只是换了个装法。
- loss 没变：都是 $|y-\hat y|=|-3.0-3.0|=6.0$。

第 4 个单元格末尾的三个 `assert` 正好证明了这一点：`assert x.shape == (2,)` 检查打包成功，`torch.allclose(y_pred_demo, y_pred_packed)` 和 `torch.allclose(loss_demo, loss_packed)` 都通过，说明两种写法的预测和损失在数值上完全相同。第 6 个单元格里把第二个输入加 1 之后，`y_pred_separate_changed` 和 `y_pred_packed_changed` 也依然相等（都从 3.0 变成 4.0），而第一个输入 `x[0]` 始终是 1.0 没有被动过。

**结论：** 打包改变的只是"数据怎么存、怎么传进函数"，不改变"算什么公式、用几个参数、算出多少"。所以选哪种写法是工程上的取舍——特征多的时候打包成一个张量更好管理（比如作业 02 里 6 个样本就适合用张量批量算），但它本身不影响模型的数学行为。